In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax
import math
import copy
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
from torch import tensor
from torch.nn.functional import pad

### Config

In [4]:
d_model=512
h=8
d_ff=2048
dropout=0.1
N=6

src_vocab, tgt_vocab = 11010, 19621
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Model

In [5]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [6]:
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [7]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x)

### Masking

In [13]:
def subsequent_mask(size):
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.uint8
    )
    return subsequent_mask == 0

In [17]:
mask = subsequent_mask(5)

In [18]:
mask

tensor([[[ True, False, False, False, False],
         [ True,  True, False, False, False],
         [ True,  True,  True, False, False],
         [ True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True]]])

### Inference

In [8]:
attn = MultiHeadedAttention(h, d_model)

In [10]:
x = torch.load("norm_result.pt")

In [14]:
x.shape

torch.Size([1, 50, 512])

In [12]:
# Encoder Inference
attn_result = attn(x, x, x, mask=None)

In [23]:
attn_result.shape

torch.Size([1, 50, 512])

In [21]:
# Decoder Inference
mask = subsequent_mask(x.shape[1])
attn_result = attn(x, x, x, mask=mask)

In [22]:
attn_result.shape

torch.Size([1, 50, 512])